1. Install dependencies

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf gradio transformers accelerate bitsandbytes langchain langchain-text-splitters accelerate bitsandbytes
print("dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
dependencies installed


2. Download a public domain book
    GENERIC BOOK DOWNLOADER (Project Gutenberg)

In [2]:
import requests
#  CHANGE THIS TO ANY GUTENBERG BOOK ID
book_id = int(input("Enter Project Gutenberg book ID: "))

# Construct the URL
url = f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt"

# Download with error handling
try:
    response = requests.get(url)
    response.encoding = 'utf-8'
    response.raise_for_status()          # Will raise an exception for 404 or other errors
    book_text = response.text

    # Save locally with a sensible filename
    filename = f"book_{book_id}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(book_text)

    print(f" Downloaded {len(book_text)} characters from book ID {book_id}")
    print(f"   Saved as: {filename}")
    print("\nFirst 500 characters:\n", book_text[:500])

except requests.exceptions.HTTPError as e:
    print(f" Failed to download book ID {book_id}. Error: {e}")
    print("   Make sure the ID exists on Project Gutenberg.")
except Exception as e:
    print(f" An error occurred: {e}")


Enter Project Gutenberg book ID: 84
 Downloaded 419434 characters from book ID 84
   Saved as: book_84.txt

First 500 characters:
 *** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chapter 11
 Chapter 12
 Chapter 13
 Chapter 14
 Chapter 15
 Chapter 16
 Chapter 17
 Chapter 18
 Chapter 19
 Chapter 20
 Chapter 21
 Chapter 22
 Chapter 23
 Chapter 24




Letter 1

_To Mrs. Saville, Engla


3. Chunk the book (intelligent splitting)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the book using the filename from Cell 2
with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

print(f" Loaded {len(text)} characters from {filename}")

# Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_text(text)

print(f"Created {len(chunks)} chunks")
if chunks:
    print("Example chunk (first 300 chars):\n", chunks[0][:300])
else:
    print("No chunks created – the file may be empty.")

 Loaded 419434 characters from book_84.txt
Created 1192 chunks
Example chunk (first 300 chars):
 *** START OF THE PROJECT GUTENBERG EBOOK 84 ***

Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chap


4. Embeddings & FAISS Index

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Check if chunks exist
try:
    chunks
except NameError:
    raise NameError("Variable 'chunks' not found. Run Cell 3 first.")

# Load embedding model (small, fast, free)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

# Create embeddings for all chunks
print(f"Creating embeddings for {len(chunks)} chunks...")
embeddings = embedder.encode(chunks, show_progress_bar=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print(f"FAISS index built with {index.ntotal} vectors")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Creating embeddings for 1192 chunks...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

FAISS index built with 1192 vectors


5. Save Index & Chunks to Disk

In [5]:
import json

# Save FAISS index
faiss.write_index(index, f"{filename.replace('.txt', '')}.index")

# Save chunks as JSON
chunks_file = f"{filename.replace('.txt', '')}_chunks.json"
with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f" Saved FAISS index to: {filename.replace('.txt', '')}.index")
print(f"Saved chunks (JSON) to: {chunks_file}")

 Saved FAISS index to: book_84.index
Saved chunks (JSON) to: book_84_chunks.json


6. Load Local LLM (phi-2 in 4‑bit)

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig
import torch

model_name = "microsoft/phi-2"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Ensure pad_token is set for tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model configuration separately
print(f"Loading configuration for {model_name}...")
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Explicitly set pad_token_id in the configuration
# This is crucial for Phi models that might not have it defined by default
# and where the model's internal PhiModel class expects it.
config.pad_token_id = tokenizer.eos_token_id

# 1. Create a BitsAndBytesConfig object and put ALL quantization settings here
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # This is now the only place to set it
    bnb_4bit_quant_type="nf4",             # Normal Float 4 (best for inference)
    bnb_4bit_compute_dtype=torch.float16,  # Keep computation in float16 for speed
    bnb_4bit_use_double_quant=True,        # Saves even more memory
)

print("Loading model in 4‑bit (this may take 1-2 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,                        # Pass the modified config here
    quantization_config=bnb_config,       # Pass the config object here
    device_map="auto",                    # Automatically offload to GPU/CPU
    trust_remote_code=True,
    torch_dtype=torch.float16,            # Use torch_dtype instead of dtype
)
print("LLM loaded successfully")

Loading tokenizer for microsoft/phi-2...
Loading configuration for microsoft/phi-2...
Loading model in 4‑bit (this may take 1-2 minutes)...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LLM loaded successfully


7. Retrieval & Generation Functions
(uses in‑memory chunks, index, embedder, model, tokenizer)

In [9]:
def retrieve(query, top_k=3):
    """Embed query, search FAISS, return top chunks."""
    q_emb = embedder.encode([query])
    distances, indices = index.search(np.array(q_emb).astype('float32'), top_k)
    retrieved = [chunks[i] for i in indices[0] if i < len(chunks)]
    return retrieved

def generate_answer(query, retrieved_chunks):
    """Build prompt and run LLM."""
    if not retrieved_chunks:
        return "I couldn't find any relevant information in the book.", []

    context = "\n\n---\n\n".join(retrieved_chunks)
    prompt = f"""Answer the question based ONLY on the context below.
If the answer isn't in the context, say "I don't have that information in the book."

Context:
{context}

Question: {query}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    full_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_answer.split("Answer:")[-1].strip()
    return answer, retrieved_chunks

# Quick test (optional)
test_query = "Who is the main character?"
retrieved = retrieve(test_query, top_k=1)
if retrieved:
    print(f"Test query: {test_query}")
    print(f"Retrieved chunk preview: {retrieved[0][:200]}...")
else:
    print("No chunks retrieved for test query.")

Test query: Who is the main character?
Retrieved chunk preview: Chapter 10...


8. Interactive Chat with Gradio
(shows book name extracted from filename)

In [10]:
import gradio as gr

# Extract a friendly book name from the filename
book_display_name = filename.replace("book_", "").replace(".txt", "")
if book_display_name.isdigit():
    # Optional: map common Gutenberg IDs to real titles
    book_names = {
        "84": "Frankenstein",
        "1342": "Pride and Prejudice",
        "345": "Dracula",
        "1661": "Sherlock Holmes",
        "11": "Alice in Wonderland"
    }
    book_display_name = book_names.get(book_display_name, f"Book #{book_display_name}")

def chat_with_book(message, history):
    retrieved = retrieve(message, top_k=3)
    answer, sources = generate_answer(message, retrieved)
    source_text = "\n\n**📖 Sources:**\n" + "\n---\n".join([s[:250] + "…" for s in sources])
    return answer + source_text

gr.ChatInterface(
    fn=chat_with_book,
    title=f"BookChat – Talk to '{book_display_name}'",
    description=f"Ask anything about {book_display_name}. Everything runs locally in Colab.",
    theme="soft"
).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a37cb2b42904816ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


9.  Quick Evaluation (Auto‑generated Questions)
(uses first few chunks to create test queries)

In [11]:
import re

print("Running a quick retrieval test...")

# Use first 5 chunks to guess character names and setting
sample_chunks = chunks[:5]
sample_text = " ".join(sample_chunks)

# Find capitalized words (possible names/places)
capitalized = set(re.findall(r'\b([A-Z][a-z]+)\b', sample_text))
common_words = {"The", "And", "For", "But", "That", "With", "From", "His", "Her", "She", "He"}
candidates = [word for word in capitalized if word not in common_words and len(word) > 2]

test_questions = []
if candidates:
    test_questions.append(f"Who is {candidates[0]}?")
test_questions.append("Where does the story begin?")
test_questions.append("What is the main conflict?")

for q in test_questions[:3]:
    retrieved = retrieve(q, top_k=1)
    print(f"\nQ: {q}")
    if retrieved:
        print(f"Retrieved chunk (first 200 chars): {retrieved[0][:200]}...")
    else:
        print("No chunks retrieved.")

Running a quick retrieval test...

Q: Who is Frankenstein?
Retrieved chunk (first 200 chars): His tale is connected and told with an appearance of the simplest truth,
yet I own to you that the letters of Felix and Safie, which he showed me,
and the apparition of the monster seen from our ship,...

Q: Where does the story begin?
Retrieved chunk (first 200 chars): the mountain. The sun is yet high in the heavens; before it descends
to hide itself behind your snowy precipices and illuminate another
world, you will have heard my story and can decide. On you it re...

Q: What is the main conflict?
Retrieved chunk (first 200 chars): mountains that in vain endeavour to emulate her; sometimes coasting the
opposite banks, we saw the mighty Jura opposing its dark side to the
ambition that would quit its native country, and an almost
...


10. Save Everything to Google Drive

In [12]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

# Create a folder named after the book
folder_name = f"BookChat_{book_display_name.replace(' ', '_')}"
drive_path = f"/content/drive/MyDrive/{folder_name}"
os.makedirs(drive_path, exist_ok=True)

# Copy files
shutil.copy(filename, drive_path)                          # original .txt
shutil.copy(f"{filename.replace('.txt', '')}.index", drive_path)   # FAISS index
shutil.copy(f"{filename.replace('.txt', '')}_chunks.json", drive_path)  # chunks (JSON)

print(f" All files saved to Google Drive at: {drive_path}")
print("Next time, you can mount Drive and reload the index & chunks (use JSON loading).")

Mounted at /content/drive
 All files saved to Google Drive at: /content/drive/MyDrive/BookChat_Frankenstein
Next time, you can mount Drive and reload the index & chunks (use JSON loading).
